# 01 — Molecular structure as a low-data learning problem

**Scientific thread**

We start from the physical object, not from the algorithm:

\[
(T,P) \longrightarrow g_{\mathrm{OO}}(r).
\]

**Goal:** understand what one training example means, inspect how molecular structure changes across thermodynamic space, and see why validation consumes part of a small information budget.

> **Workshop rhythm:** run the short experiment cells, inspect the figure, then discuss the questions before moving on.

In [ ]:
#@title 0. Workshop setup — run once { display-mode: "form" }
# This cell intentionally hides infrastructure so workshop time stays focused on physics.

from pathlib import Path
import hashlib, importlib.util, os, shutil, subprocess, sys, urllib.request, zipfile

ASSET_URL = "" #@param {type:"string"}
EXPECTED_ASSET_SHA256 = "2e75fd65ad39a9dec41f7b089c2aafe51a6bb14eeee049b055be8ea3953a7bea"
WORKSHOP_ROOT = Path("/content/ThermoRDF-Workshop")
ASSET_NAME = "ThermoRDF-Colab-Assets.zip"

# Local/instructor execution override used only for automated testing.
_local_root = os.environ.get("THERMORDF_WORKSHOP_ROOT", "").strip()
if _local_root:
    WORKSHOP_ROOT = Path(_local_root).resolve()
else:
    ready = (WORKSHOP_ROOT / "data/teaching/stage_02_b48_train_40.csv.gz").is_file()
    if not ready:
        archive = Path("/content") / ASSET_NAME
        if ASSET_URL.strip():
            print("Downloading workshop assets ...")
            urllib.request.urlretrieve(ASSET_URL.strip(), archive)
        else:
            try:
                from google.colab import files
            except ImportError as exc:
                raise RuntimeError("This notebook is configured for Google Colab. Set THERMORDF_WORKSHOP_ROOT for local testing.") from exc
            print(f"Upload the companion file: {ASSET_NAME}")
            uploaded = files.upload()
            if ASSET_NAME not in uploaded:
                raise RuntimeError(f"Expected {ASSET_NAME}. Please rerun this cell and upload that file.")
            archive.write_bytes(uploaded[ASSET_NAME])

        digest = hashlib.sha256(archive.read_bytes()).hexdigest()
        if digest != EXPECTED_ASSET_SHA256:
            raise RuntimeError("Asset bundle checksum mismatch. Use the bundle distributed with these notebooks.")

        if WORKSHOP_ROOT.exists():
            shutil.rmtree(WORKSHOP_ROOT)
        WORKSHOP_ROOT.mkdir(parents=True)
        with zipfile.ZipFile(archive) as zf:
            zf.extractall(WORKSHOP_ROOT)

_required = {"numpy":"numpy", "pandas":"pandas", "matplotlib":"matplotlib", "scikit-learn":"sklearn", "torch":"torch"}
_missing = [pkg for pkg, module in _required.items() if importlib.util.find_spec(module) is None]
if _missing:
    print("Installing missing Colab packages:", ", ".join(_missing))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *_missing])

sys.path.insert(0, str(WORKSHOP_ROOT / "src"))
from thermordf_workshop import *
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

torch.set_num_threads(min(2, os.cpu_count() or 1))
data = load_workshop_data(WORKSHOP_ROOT)
print(f"Workshop ready | {len(data.train)} training + {len(data.validation)} validation states | {len(data.r_nm)} RDF coordinates")

## Physical question — what exactly is scarce?

A thermodynamic state gives us **one structured molecular observation**: an RDF sampled at 725 radial coordinates.

Those 725 values are strongly correlated parts of the same curve. They are **not 725 independent training examples**.

In [ ]:
print(f"training states   : {len(data.train)}")
print(f"validation states : {len(data.validation)}")
print(f"RDF coordinates   : {len(data.r_nm)}")

## Experiment 1 — look at the structure before learning it

The curves below come only from the development training set. Focus on the first peak, first minimum, subsequent oscillations and relaxation towards the bulk value.

In [ ]:
plot_representative_rdfs(data);

### Observe

- Which RDF features change most visibly with thermodynamic state?
- Which features mostly change in amplitude, and which shift in position?
- Which radial regions look smooth enough to be learnable from nearby states?
- Why is predicting the **entire curve** harder than predicting a single scalar?

## Experiment 2 — allocate a small simulation budget

The thermodynamic domain is broad, but the development budget contains only **48 molecular states**. Forty states provide fitting information and eight are withheld for development validation.

In [ ]:
plot_development_budget(data);

In [ ]:
data.validation[["state_id", "T_K", "P_bar"]].reset_index(drop=True)

### Interpret

A held-out state is **not free information**. Every validation state is one molecular simulation that cannot simultaneously be used to fit the model.

\[
\boxed{\text{learn a thermodynamic–structural map from deliberately few states}}
\]

**Take-away:** the low-data condition is defined by the number and placement of independent thermodynamic states—not by the 725 coordinates used to represent each RDF.